# Travel Lab — ממשק Streamlit

פתחי את המחברת ב־Google Colab ובחרי **Runtime → Run all**. ברירת המחדל טוענת את הקוד והדאטא מ־GitHub (`SOURCE_MODE = "github"`); אפשר לבחור `drive` כדי לטעון מתיקיית `travel` בדרייב. בתא האחרון יופיע הממשק: בחרי תרחיש ולחצי **הרץ השוואה**. ברירת המחדל לסוכנים היא Gemini 3.1 Flash-Lite; המפתח נטען מ־Colab Secrets בשם `GEMINI_API_KEY`. ללא מפתח אפשר להריץ את מנוע הקוד.

המחברת קטנה; היא קוראת את הקוד ואת הדאטא משיבוט של המאגר או מתיקיית `travel` בדרייב. אין צורך להעלות קבצים מחדש ואין צורך ב־GPU. זו סביבת הדגמה זמנית: השאירי את Colab מחובר ושמרי את התוצאות דרך כפתורי ההורדה בממשק.

עדכון משימה 2: בחיבור GitHub, תא ההפעלה מרענן את חבילת הקוד כולה. הגדרות Gemini המשותפות תואמות למחברת ההוכחה, כולל בקרת הקצב. הצלחת ההרצה הראשית מול Gemini עדיין טעונה אימות.


## 1. חיבור וטעינה

`SOURCE_MODE = "github"` (ברירת מחדל): שיבוט המאגר ל־`/content/Travel_Agent`; הקוד בשורש המאגר והדאטא ב־`data/`. מחוץ ל־Colab, אם המחברת רצה מתוך שיבוט מקומי, נעשה שימוש בו.

`SOURCE_MODE = "drive"`: `My Drive/travel` ובה `project` ו־`data`. נוצר עותק עבודה זמני; קבצי המקור נשארים כפי שהם.


In [ ]:
#@title טעינת הפרויקט והדאטא
SOURCE_MODE = "github" #@param ["github", "drive"]
GITHUB_REPO = "https://github.com/Noven2155/Travel_Agent.git" #@param {type:"string"}
GITHUB_BRANCH = "main" #@param {type:"string"}
DRIVE_TRAVEL_FOLDER = "/content/drive/MyDrive/travel" #@param {type:"string"}

import hashlib, importlib.util, json, os, shutil, subprocess, sys
from pathlib import Path

IN_COLAB = bool(importlib.util.find_spec("google.colab")) if importlib.util.find_spec("google") else False

def _find_local_repo():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").is_file() and (folder / "streamlit_app.py").is_file():
            return folder
    return None

if SOURCE_MODE == "drive":
    # Drive layout: travel/project (code) and travel/data (data) side by side.
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    SOURCE = Path(os.environ.get("TRAVEL_DRIVE_FOLDER", DRIVE_TRAVEL_FOLDER))
else:
    # Repo layout: code at the repository root, data/ beside it.
    SOURCE = None if IN_COLAB else _find_local_repo()
    if SOURCE is None:
        SOURCE = Path("/content/Travel_Agent" if IN_COLAB else "Travel_Agent_clone").absolute()
        if (SOURCE / ".git").is_dir():
            subprocess.run(["git", "-C", str(SOURCE), "pull", "--ff-only"], check=True)
        else:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(SOURCE)], check=True)
PROJECT_SRC = SOURCE / "project" if (SOURCE / "project").is_dir() else SOURCE
DATA_SRC = SOURCE / "data"
REPO_LAYOUT = PROJECT_SRC == SOURCE
print("מקור הקוד:", PROJECT_SRC, "| מקור הדאטא:", DATA_SRC)

if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("נדרש Python 3.10–3.13. אפשר לבחור סביבת Colab עם Python 3.12.")
WORK = Path(os.environ.get("TRAVEL_STREAMLIT_WORK", "/content/travel_streamlit" if IN_COLAB else "travel_streamlit_runtime"))
PROJECT = WORK / "project"
for required in (PROJECT_SRC / "streamlit_app.py", PROJECT_SRC / "requirements-streamlit.txt", PROJECT_SRC / "src/travel_agent/streamlit_view.py", DATA_SRC / "inventory.json"):
    if not required.is_file():
        raise FileNotFoundError(f"חסר {required}. ודאי שהקוד והדאטא העדכניים זמינים (GitHub או תיקיית travel בדרייב).")
PROJECT.mkdir(parents=True, exist_ok=True)
for relative in ("src", "web", ".streamlit", "frontend/public/fonts"):
    source = PROJECT_SRC / relative
    if source.is_dir():
        shutil.copytree(source, PROJECT / relative, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
for name in ("streamlit_app.py", "requirements-streamlit.txt"):
    shutil.copy2(PROJECT_SRC / name, PROJECT / name)
for relative in ("places", "weather"):
    shutil.copytree(DATA_SRC / relative, WORK / "data" / relative, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
for name in ("inventory.json", "README_COMPARISON_DATA_HE.md"):
    shutil.copy2(DATA_SRC / name, WORK / "data" / name)
expected = "39a0c9c2b2562062211d884af35cc73b3826926b0c56cb06ea5e428c87167319"
if hashlib.sha256((WORK / "data/inventory.json").read_bytes()).hexdigest() != expected:
    raise RuntimeError("הקטלוג שונה מזה שנבדק. נדרשת בדיקת התרחישים מול הקטלוג המעודכן.")
catalog = json.loads((WORK / "data/inventory.json").read_text())
print("הפרויקט נטען. טיסות:", len(catalog["flights"]), "לינות:", len(catalog["stays"]), "רכבים:", len(catalog["cars"]))


## 2. התקנת הספריות

אם ההתקנה כבר הסתיימה בסביבה הנוכחית, דלגי על התא הזה ועברי לפתיחת הממשק. בזמן התקנה יוצגו זמן שחלף והשלב האחרון כל עשר שניות.

In [ ]:
#@title התקנת הספריות — אפשר לדלג אם כבר הותקנו
import runpy
helper = PROJECT_SRC / "scripts/notebook_install.py"
runpy.run_path(str(helper))["install"](PROJECT / "requirements-streamlit.txt", WORK / "install.log")


## 3. פתיחת הממשק — חיבור אוטומטי ל־Secrets

אם הספריות כבר הותקנו, הריצי רק את התא הבא. הוא קורא את GEMINI_API_KEY מ־Colab Secrets ומעביר אותו לשרת Streamlit. ודאי שמתג Notebook access מופעל עבור הסוד הזה. אין צורך להזין את המפתח שוב בממשק. תוכן המפתח אינו מודפס.


In [ ]:
#@title Start Streamlit with current source and secure model settings
import runpy
import shutil
import subprocess

if "PROJECT" not in globals():
    raise RuntimeError("Run the source/data cell first.")
if SOURCE_MODE == "github" and (SOURCE / ".git").is_dir():
    subprocess.run(["git", "-C", str(SOURCE), "pull", "--ff-only"], check=True)

# Refresh the entire package so newly added provider modules are included.
package_source = PROJECT_SRC / "src/travel_agent"
if not package_source.is_dir():
    raise FileNotFoundError("The source package is missing; reload the project source.")
shutil.copytree(package_source, PROJECT / "src/travel_agent", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
scenario_source = PROJECT_SRC / "web/scenarios.json"
scenario_target = PROJECT / "web/scenarios.json"
scenario_target.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(scenario_source, scenario_target)
print("Current package and scenario definitions are ready.", flush=True)

helper = PROJECT_SRC / "scripts/notebook_server.py"
if not helper.is_file():
    raise FileNotFoundError("The server helper is missing from the selected source.")

def _read_model_secret(secret_store):
    try:
        value = secret_store.get("GEMINI_API_KEY")
    except Exception:
        raise RuntimeError("Enable Notebook access for GEMINI_API_KEY in Colab Secrets.") from None
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError("GEMINI_API_KEY is empty in Colab Secrets.")
    return {"GEMINI_API_KEY": value.strip()}

_model_env = dict(LLM_TEMPERATURE="0", LLM_SEED="42", LLM_MAX_TOKENS="8192",
                  LLM_TIMEOUT_S="60", LLM_RPM="12")
if IN_COLAB:
    from google.colab import userdata
    _model_env.update(_read_model_secret(userdata))
    print("Gemini credentials loaded securely. Provider availability is checked during execution.", flush=True)

STREAMLIT_PROCESS, STREAMLIT_PORT = runpy.run_path(str(helper))["start_server"](
    PROJECT, WORK, previous=globals().get("STREAMLIT_PROCESS"), colab=IN_COLAB, env=_model_env)
_model_env.clear()
if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_iframe(STREAMLIT_PORT, height=1150, cache_in_notebook=False)
else:
    print(f"Open http://localhost:{STREAMLIT_PORT}")


## מה נבדק ומה עוד נדרש

מנוע הקוד נבדק בפועל דרך ממשק Streamlit בכל ארבעת התרחישים. תוצאות B/C הן תוצאות הרצה רק לאחר חיבור מודל תקין; ללא מפתח הן מסומנות כטרם הופעלו. זמינות ותאריכים בדאטא כוללים הדמיה, ואין הזמנות אמיתיות.

טעינת הקבצים ופקדי הממשק נבדקים מקומית. אישור הדרייב, תצוגת המסגרת והרצת Gemini אמיתי עדיין דורשים בדיקה בחשבון Colab שלך. אם לא מופיע ממשק, ודאי שתא ההתקנה ותא ההפעלה הסתיימו בהצלחה. להורדה למחשב או לקישור צוות קבוע: `docs/README_STREAMLIT_HE.md`.

מקורות: [ממשק הפלט של Colab](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py), [Streamlit Community Cloud](https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/deploy).

בעת הפעלה מחדש של התא האחרון, קובץ התרחישים (`web/scenarios.json`) מועתק מחדש מהמקור (שיבוט המאגר או ה־Drive) לפני הפעלת Streamlit.